In [1]:
from pyspark.sql import functions as F
from pyspark.sql import types as T

StatementMeta(, 28f46cbd-8766-4bd1-a205-6529d65edf73, 3, Finished, Available, Finished, False)

In [2]:
# ---------------------------------------------------------
# Config: optional optimizations for Delta writes (good practice)
# ---------------------------------------------------------
spark.conf.set("spark.sql.shuffle.partitions", "200")
spark.conf.set("spark.sql.sources.partitionOverwriteMode", "dynamic")

StatementMeta(, 28f46cbd-8766-4bd1-a205-6529d65edf73, 4, Finished, Available, Finished, False)

## 1. Silver sales_daily from bronze_train_sales

In [3]:
df_train = spark.table("bronze_train_sales")

# Cast and basic cleaning
df_sales_daily = (
    df_train
    .withColumn("date", F.to_date("date"))  # ensure date type
    .withColumn("store_nbr", F.col("store_nbr").cast(T.IntegerType()))
    .withColumn("onpromotion", F.col("onpromotion").cast(T.IntegerType()))
    .withColumn("sales", F.col("sales").cast(T.DoubleType()))
    .dropna(subset=["date", "store_nbr", "family", "sales"])  # drop impossible nulls
)

StatementMeta(, 28f46cbd-8766-4bd1-a205-6529d65edf73, 5, Finished, Available, Finished, False)

In [4]:
# remove negative sales if any
df_sales_daily = df_sales_daily.filter(F.col("sales") >= 0)

# Write as Silver table
df_sales_daily.write.mode("overwrite").format("delta").saveAsTable("silver_sales_daily")

StatementMeta(, 28f46cbd-8766-4bd1-a205-6529d65edf73, 6, Finished, Available, Finished, False)

## 2. Silver store_dim from bronze_stores

In [5]:
df_stores = spark.table("bronze_stores")

df_store_dim = (
    df_stores
    .withColumn("store_nbr", F.col("store_nbr").cast(T.IntegerType()))
    .select(
        "store_nbr",
        "city",
        "state",
        "type",
        "cluster"
    )
    .dropDuplicates(["store_nbr"])
)

df_store_dim.write.mode("overwrite").format("delta").saveAsTable("silver_store_dim")

StatementMeta(, 28f46cbd-8766-4bd1-a205-6529d65edf73, 7, Finished, Available, Finished, False)

In [8]:
display(df_store_dim)

StatementMeta(, 28f46cbd-8766-4bd1-a205-6529d65edf73, 10, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, a9d4d9e9-791a-4180-af7c-ac6cbd1865d7)

## 3. Silver item_dim (from distinct families in train)

In [9]:
df_item_dim = (
    df_sales_daily
    .select("family")
    .dropDuplicates()
    .withColumn("item_id", F.monotonically_increasing_id())  # simple surrogate key
    .select("item_id", "family")
)

df_item_dim.write.mode("overwrite").format("delta").saveAsTable("silver_item_dim")

StatementMeta(, 28f46cbd-8766-4bd1-a205-6529d65edf73, 11, Finished, Available, Finished, False)

## 4. Silver calendar_dim from holidays_events plus generic calendar

In [10]:
df_holidays = spark.table("bronze_holidays_events")

df_holidays_clean = (
    df_holidays
    .withColumn("date", F.to_date("date"))
    .withColumn("is_transferred", F.col("transferred").cast(T.BooleanType()))
    .select(
        "date",
        "type",
        "locale",
        "locale_name",
        "description",
        "is_transferred"
    )
)

StatementMeta(, 28f46cbd-8766-4bd1-a205-6529d65edf73, 12, Finished, Available, Finished, False)

In [11]:
# Build a continuous calendar from min to max date in sales
date_bounds = df_sales_daily.agg(
    F.min("date").alias("min_date"),
    F.max("date").alias("max_date")
).collect()[0]

min_date = date_bounds["min_date"]
max_date = date_bounds["max_date"]


StatementMeta(, 28f46cbd-8766-4bd1-a205-6529d65edf73, 13, Finished, Available, Finished, False)

In [15]:
# Use sequence() and explode() ->for generating a continuous date series

calendar_df = (
    spark.sql(f"SELECT sequence(to_date('{min_date}'), to_date('{max_date}'), interval 1 day) AS date_array")
    .select(F.explode("date_array").alias("date"))
)

# sequence(start, end, interval 1 day) creates an array of dates from min_date to max_date.
# explode turns that array into one row per date.

StatementMeta(, 28f46cbd-8766-4bd1-a205-6529d65edf73, 17, Finished, Available, Finished, False)

In [16]:
# calendar attriutes

calendar_df = (
    calendar_df
    .withColumn("year", F.year("date"))
    .withColumn("month", F.month("date"))
    .withColumn("day", F.dayofmonth("date"))
    .withColumn("day_of_week", F.date_format("date", "E"))
    .withColumn("weekofyear", F.weekofyear("date"))
    .withColumn("is_month_start", F.dayofmonth("date") == 1)
    .withColumn("is_month_end", F.last_day("date") == F.col("date"))
)

StatementMeta(, 28f46cbd-8766-4bd1-a205-6529d65edf73, 18, Finished, Available, Finished, False)

In [17]:
# Join holiday info (left join – not every day is a holiday)
df_calendar_dim = (
    calendar_df
    .join(
        df_holidays_clean,
        on="date",
        how="left"
    )
    .withColumn(
        "is_holiday",
        F.when(F.col("type").isNotNull(), F.lit(True)).otherwise(F.lit(False))
    )
)

df_calendar_dim.write.mode("overwrite").format("delta").saveAsTable("silver_calendar_dim")

StatementMeta(, 28f46cbd-8766-4bd1-a205-6529d65edf73, 19, Finished, Available, Finished, False)

In [ ]:
# What this code does

# Reads bronze tables using spark.table("bronze_*").

# Cleans and casts types to appropriate numeric/date types.

# Creates:

# silver_sales_daily – clean daily sales with proper types and no null/negative sales.

# silver_store_dim – store dimension from metadata.

# silver_item_dim – dimension for the product families.

# silver_calendar_dim – full calendar between min/max sales date with calendar attributes and holiday flags from holidays_events.

